# ItemKNN Baseline dưới thiết lập Leave-One-Out

Notebook này triển khai **Item-based KNN (ItemKNN)** như một baseline collaborative filtering cổ điển để so sánh với LightFM baseline và phương pháp đề xuất. Điểm số gợi ý được xây dựng từ **cosine similarity** trên ma trận tương tác nhị phân giữa người dùng và item.

## Cấu hình phương pháp

| Thành phần | Mô tả |
|-----------|----------|
| **Ma trận đầu vào** | Ma trận user-item nhị phân (`rate >= POSITIVE_THRESHOLD` được xem là tương tác dương) |
| **Độ tương đồng** | Cosine similarity giữa các vector item |
| **Lân cận** | Top-K nearest neighbors |
| **Hàm chấm điểm** | Tổng độ tương đồng của các item lân cận mà người dùng đã tương tác |
| **Đặc trưng bổ sung** | Không sử dụng metadata, đặc trưng nội dung hoặc side information |
| **Chuẩn hóa điểm** | Không áp dụng |
| **Shrinkage** | Không áp dụng |

## Các thành phần được giữ nhất quán với LightFM baseline
- Cùng bộ dữ liệu đánh giá.
- Cùng thiết lập chia dữ liệu Leave-One-Out (LOO) theo người dùng.
- Cùng định nghĩa tương tác nhị phân (`rate >= THRESHOLD`).
- Cùng phân tách item warm/cold trên tập kiểm thử.
- Cùng hệ chỉ số đánh giá: Precision, Recall, NDCG, HR, MRR @K.
- Cùng nguyên tắc loại bỏ các item đã xuất hiện trong train khi xếp hạng.

## Hạn chế phương pháp
- **Cold-start item:** item chưa xuất hiện trong train sẽ không có vector tương tác, do đó không thể được chấm điểm.
- **Cold-start user:** người dùng không có lịch sử tương tác sẽ không thể nhận gợi ý theo cơ chế collaborative filtering thuần túy.

---

## Mục 0 — Thiết lập môi trường và cấu hình

In [1]:
import os, warnings
from collections import Counter

import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import norm as sparse_norm

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Dataset URLs ───────────────────────────────────────────────────────────────
MOVIES_URL  = "https://raw.githubusercontent.com/lynchblue/movie-rating-dataset/main/data/movies.csv"
RATINGS_URL = "https://raw.githubusercontent.com/lynchblue/movie-rating-dataset/main/data/ratings.csv"
USERS_URL   = "https://raw.githubusercontent.com/lynchblue/movie-rating-dataset/main/data/user_profiles.csv"

# ── Configuration ──────────────────────────────────────────────────────────────
POSITIVE_THRESHOLD = 7       # Binary implicit: rate >= này → positive
MIN_RATINGS        = 10      # Lọc user có ít hơn N ratings
K_NEIGHBOURS       = 50      # Số lân cận gần nhất để tính score
K_VALUES           = (5, 10, 20, 50)

print(f"[ItemKNN] Config: threshold={POSITIVE_THRESHOLD}, K_neighbours={K_NEIGHBOURS}")
print(f"[ItemKNN] Split : Leave-One-Out (LOO) per user")
print(f"[ItemKNN] Similarity: Cosine (binary implicit matrix)")
print(f"[ItemKNN] NO content features, NO side information")

[ItemKNN] Config: threshold=7, K_neighbours=50
[ItemKNN] Split : Leave-One-Out (LOO) per user
[ItemKNN] Similarity: Cosine (binary implicit matrix)
[ItemKNN] NO content features, NO side information


## Mục 0b — Các hàm hỗ trợ

In [2]:
def cleanup_column(series):
    """Fix encoding artefacts in pipe-separated string Series."""
    replacements = {
        '| ': '|', '\xc3\xa9': 'é', '\xc3\x81': 'Á', '\xc3\x93': 'Ó',
        '\xc3\xa1': 'á', '\xc3\xb3': 'ó', '\xc3\xb1': 'ñ', '\xc3\xad': 'í',
        '\xc3\xba': 'ú', 'Ã©': 'é', 'Ã¡': 'á', 'Ã³': 'ó', 'Ã±': 'ñ',
        'Ãº': 'ú',
    }
    col = series.copy()
    for bad, good in replacements.items():
        col = col.str.replace(bad, good, regex=False)
    return col

print("Helpers defined: cleanup_column")

Helpers defined: cleanup_column


## Mục 1 — Nạp dữ liệu

In [3]:
ratings_df = pd.read_csv(RATINGS_URL)
users_df   = pd.read_csv(USERS_URL)
movies_df  = pd.read_csv(MOVIES_URL)

ratings_df["date"] = pd.to_datetime(ratings_df["date"])

print(f"Users   : {users_df['id'].nunique()}")
print(f"Movies  : {movies_df['id'].nunique()}")
print(f"Ratings : {len(ratings_df):,}")
print(f"Rating range: {ratings_df['rate'].min()} – {ratings_df['rate'].max()}")
print(f"Date range  : {ratings_df['date'].min().date()} – {ratings_df['date'].max().date()}")

Users   : 482
Movies  : 78628
Ratings : 1,172,038
Rating range: 1 – 10
Date range  : 2002-08-14 – 2021-04-01


## Mục 2 — Lọc dữ liệu hợp lệ

In [4]:
# 2.1 Lọc orphan users (có rating nhưng KHÔNG có profile)
users_in_profile = set(users_df["id"].unique())
ratings_df = ratings_df[ratings_df["id"].isin(users_in_profile)].copy()
print(f"Sau lọc orphan users    : {len(ratings_df):,} ratings")

# 2.2 Lọc user có ít hơn MIN_RATINGS ratings
rating_counts = ratings_df.groupby("id")["movie_id"].count()
valid_users   = rating_counts[rating_counts >= MIN_RATINGS].index
ratings_df    = ratings_df[ratings_df["id"].isin(valid_users)].copy()
users_df      = users_df[users_df["id"].isin(valid_users)].copy()
print(f"Sau lọc min {MIN_RATINGS} ratings : {len(ratings_df):,} ratings, {ratings_df['id'].nunique()} users")

# 2.3 Lọc movie không tồn tại trong movies.csv
movies_in_table = set(movies_df["id"].unique())
ratings_df = ratings_df[ratings_df["movie_id"].isin(movies_in_table)].copy()
print(f"Sau lọc missing movies  : {len(ratings_df):,} ratings")

Sau lọc orphan users    : 963,266 ratings
Sau lọc min 10 ratings : 963,231 ratings, 474 users
Sau lọc missing movies  : 963,228 ratings


## Mục 3 — Chuẩn hóa các cột văn bản

In [5]:
for col in ["country_name", "genres", "topics", "directors", "actors",
            "script", "producer", "music", "photography"]:
    movies_df[col] = cleanup_column(movies_df[col])

print("Text cleanup hoàn tất.")

Text cleanup hoàn tất.


## Mục 4 — Chia dữ liệu theo Leave-One-Out (LOO)

Với mỗi user, rating **mới nhất** → test, rating **kế tiếp** → valid,
toàn bộ còn lại → train.

> Điều kiện `MIN_RATINGS >= 3` đã được bảo đảm ở Mục 2.

In [6]:
ratings_df = ratings_df.sort_values(["id", "date"]).reset_index(drop=True)

train_idx = []
valid_idx = []
test_idx  = []

for uid, group in ratings_df.groupby("id", sort=False):
    idx = group.index.tolist()
    if len(idx) < 3:
        train_idx.extend(idx)
        continue
    test_idx.append(idx[-1])
    valid_idx.append(idx[-2])
    train_idx.extend(idx[:-2])

train_df = ratings_df.loc[train_idx].copy().reset_index(drop=True)
valid_df = ratings_df.loc[valid_idx].copy().reset_index(drop=True)
test_df  = ratings_df.loc[test_idx].copy().reset_index(drop=True)

print(f"Train : {len(train_df):>7,} ratings  |  {train_df['id'].nunique()} users")
print(f"Valid : {len(valid_df):>7,} ratings  |  {valid_df['id'].nunique()} users  (1 rating/user)")
print(f"Test  : {len(test_df):>7,} ratings  |  {test_df['id'].nunique()} users  (1 rating/user)")
print(f"\nDate range train: {train_df['date'].min().date()} – {train_df['date'].max().date()}")
print(f"Date range valid: {valid_df['date'].min().date()} – {valid_df['date'].max().date()}")
print(f"Date range test : {test_df['date'].min().date()} – {test_df['date'].max().date()}")

Train : 962,280 ratings  |  474 users
Valid :     474 ratings  |  474 users  (1 rating/user)
Test  :     474 ratings  |  474 users  (1 rating/user)

Date range train: 2002-08-14 – 2021-03-31
Date range valid: 2011-12-06 – 2021-04-01
Date range test : 2011-12-08 – 2021-04-01


## Mục 5 — Phân loại item warm và cold trên tập kiểm thử

- **Warm items:** movie đã xuất hiện trong train → có interaction vector → itemKNN có thể score.
- **Cold items:** movie **không** có trong train → **itemKNN không thể gợi ý** (không có vector để tính similarity).

> Đây là hạn chế cố hữu của collaborative filtering thuần túy.

In [7]:
movies_in_train = set(train_df["movie_id"].unique())

test_warm_mask = test_df["movie_id"].isin(movies_in_train)
test_cold_mask = ~test_warm_mask

test_warm_df = test_df[test_warm_mask].copy().reset_index(drop=True)
test_cold_df = test_df[test_cold_mask].copy().reset_index(drop=True)

print(f"Unique items in train : {len(movies_in_train):,}")
print(f"Unique items in valid : {valid_df['movie_id'].nunique():,}")
print(f"Unique items in test  : {test_df['movie_id'].nunique():,}")
print()
print(f"TEST total : {len(test_df):>6,} ratings")
print(f"TEST warm  : {len(test_warm_df):>6,} ratings  "
      f"({len(test_warm_df)/len(test_df)*100:.1f}%)  "
      f"— {test_warm_df['movie_id'].nunique():,} unique movies")
print(f"TEST cold  : {len(test_cold_df):>6,} ratings  "
      f"({len(test_cold_df)/len(test_df)*100:.1f}%)  "
      f"— {test_cold_df['movie_id'].nunique():,} unique movies")
print()
print("⚠ ItemKNN sẽ trả về score=0 cho tất cả cold items → TEST_COLD metrics = 0.0")

Unique items in train : 75,115
Unique items in valid : 398
Unique items in test  : 407

TEST total :    474 ratings
TEST warm  :    446 ratings  (94.1%)  — 379 unique movies
TEST cold  :     28 ratings  (5.9%)  — 28 unique movies

⚠ ItemKNN sẽ trả về score=0 cho tất cả cold items → TEST_COLD metrics = 0.0


## Mục 6 — Xây dựng ánh xạ chỉ mục

Tạo ánh xạ `user_id ↔ row_index` và `movie_id ↔ col_index` cho ma trận sparse.

In [8]:
# Tập hợp toàn bộ users và items trong tập train
all_users  = sorted(train_df["id"].unique().tolist())
all_movies = sorted(train_df["movie_id"].unique().tolist())

user2idx  = {u: i for i, u in enumerate(all_users)}
movie2idx = {m: i for i, m in enumerate(all_movies)}
idx2movie = {i: m for m, i in movie2idx.items()}
idx2user  = {i: u for u, i in user2idx.items()}

n_users  = len(all_users)
n_movies = len(all_movies)

print(f"Users trong train  : {n_users:,}")
print(f"Movies trong train : {n_movies:,}")

Users trong train  : 474
Movies trong train : 75,115


## Mục 7 — Xây dựng ma trận user-item nhị phân

Ma trận `R` shape `(n_users, n_movies)` với:
- `R[u, i] = 1` nếu user u đã rate movie i với `rate >= POSITIVE_THRESHOLD`
- `R[u, i] = 0` otherwise

> Giá trị rating gốc không được sử dụng; chỉ tín hiệu nhị phân được giữ lại để bảo đảm tính nhất quán với LightFM baseline.

In [9]:
# Chỉ lấy positive interactions (rate >= threshold)
train_positive = train_df[train_df["rate"] >= POSITIVE_THRESHOLD].copy()

# Lọc các user/movie có trong mapping
train_positive = train_positive[
    train_positive["id"].isin(user2idx) &
    train_positive["movie_id"].isin(movie2idx)
]

row_indices = train_positive["id"].map(user2idx).values
col_indices = train_positive["movie_id"].map(movie2idx).values
data        = np.ones(len(train_positive), dtype=np.float32)

# Ma trận user-item (binary)
R = csr_matrix(
    (data, (row_indices, col_indices)),
    shape=(n_users, n_movies),
    dtype=np.float32,
)

print(f"User-Item matrix shape : {R.shape}")
print(f"Positive interactions  : {R.nnz:,}")
print(f"Matrix density         : {R.nnz / (n_users * n_movies) * 100:.4f}%")
print(f"Threshold              : rate >= {POSITIVE_THRESHOLD}")

User-Item matrix shape : (474, 75115)
Positive interactions  : 412,438
Matrix density         : 1.1584%
Threshold              : rate >= 7


## Mục 8 — Tính độ tương đồng cosine giữa các item

Cosine similarity giữa hai item $i$ và $j$ được tính trên cột của ma trận $R^T$:

$$\text{sim}(i, j) = \frac{\mathbf{r}_i \cdot \mathbf{r}_j}{\|\mathbf{r}_i\| \cdot \|\mathbf{r}_j\|}$$

**Triển khai:**
1. Chuẩn hóa L2 từng cột (= item vector)
2. Ma trận similarity = $R_{norm}^T \cdot R_{norm}$ (matrix multiplication)

> Không xây dựng toàn bộ ma trận độ tương đồng kích thước đầy đủ do chi phí bộ nhớ lớn.
> Thay vào đó, độ tương đồng được tính theo nhu cầu tại thời điểm suy luận.

In [10]:
from sklearn.preprocessing import normalize

# Chuẩn hóa L2 theo cột (mỗi item là 1 cột trong R)
# Chuyển sang item-user: shape (n_movies, n_users)
R_T = R.T.tocsr()  # shape: (n_movies, n_users)

# Normalize L2 mỗi item vector (mỗi hàng của R_T)
R_T_norm = normalize(R_T, norm='l2', axis=1)  # shape: (n_movies, n_users)

print(f"Item-User matrix (R_T) shape : {R_T.shape}")
print(f"Normalized R_T shape         : {R_T_norm.shape}")
print(f"[ItemKNN] Similarity sẽ được tính on-the-fly per user tại inference")
print(f"[ItemKNN] K_NEIGHBOURS = {K_NEIGHBOURS}")

Item-User matrix (R_T) shape : (75115, 474)
Normalized R_T shape         : (75115, 474)
[ItemKNN] Similarity sẽ được tính on-the-fly per user tại inference
[ItemKNN] K_NEIGHBOURS = 50


## Mục 9 — Hàm chấm điểm ItemKNN

**Nguyên lý chấm điểm:**

Với user $u$ và item ứng viên $j$:

$$\text{score}(u, j) = \sum_{i \in N(j, K) \cap I_u^+} \text{sim}(i, j)$$

Trong đó:
- $N(j, K)$ = K nearest neighbors của item $j$ (theo cosine similarity)
- $I_u^+$ = tập item mà user $u$ đã tương tác tích cực trong train

**Các thành phần không được sử dụng trong baseline này:**
- Shrinkage hoặc normalization
- Cơ chế weighting decay theo độ phổ biến
- Các heuristic bổ sung khác

In [11]:
def get_user_scores_itemknn(user_idx, R, R_T_norm, k_neighbours=K_NEIGHBOURS):
    """
    Tính score itemKNN cho TẤT CẢ items trong catalogue với 1 user.

    Parameters
    ----------
    user_idx   : int — row index của user trong R
    R          : csr_matrix (n_users, n_movies) — binary interaction matrix
    R_T_norm   : csr_matrix (n_movies, n_users) — L2-normalized item-user matrix
    k_neighbours: int — số lân cận dùng để tính score

    Returns
    -------
    scores : np.ndarray shape (n_movies,) — score cho từng item
    """
    n_movies = R_T_norm.shape[0]

    # Lấy tập items user đã tương tác (positive) trong train
    user_row   = R[user_idx]              # shape (1, n_movies)
    user_items = user_row.indices         # index của items đã tương tác

    if len(user_items) == 0:
        return np.zeros(n_movies, dtype=np.float32)

    # Lấy item vectors của các items user đã tương tác
    # shape: (n_interacted, n_users)
    interacted_vecs = R_T_norm[user_items]  # sparse

    # Tính similarity giữa MỌI items trong catalogue và các items user đã tương tác
    # sim_matrix shape: (n_movies, n_interacted)
    # = R_T_norm (n_movies, n_users) @ interacted_vecs.T (n_users, n_interacted)
    sim_matrix = R_T_norm.dot(interacted_vecs.T)  # (n_movies, n_interacted)

    # Chuyển về dense
    sim_dense = np.asarray(sim_matrix.todense(), dtype=np.float32)  # (n_movies, n_interacted)

    # Với mỗi candidate item, chỉ giữ Top-K neighbors trong số các items đã tương tác
    # Nếu n_interacted <= k_neighbours: giữ tất cả
    if sim_dense.shape[1] > k_neighbours:
        # Sắp xếp theo chiều ngang, lấy top-K similarity
        top_k_idx  = np.argpartition(-sim_dense, k_neighbours, axis=1)[:, :k_neighbours]
        rows       = np.arange(n_movies)[:, None]
        sim_top_k  = sim_dense[rows, top_k_idx]
        scores     = sim_top_k.sum(axis=1)
    else:
        scores = sim_dense.sum(axis=1)  # (n_movies,)

    return scores


print("[ItemKNN] Scoring function defined: get_user_scores_itemknn")
print(f"          Neighbourhood size K = {K_NEIGHBOURS}")

[ItemKNN] Scoring function defined: get_user_scores_itemknn
          Neighbourhood size K = 50


## Mục 10 — Các chỉ số đánh giá: Precision, Recall, NDCG, HR, MRR @K

Công thức đánh giá được giữ đồng nhất với LightFM baseline để bảo đảm tính so sánh công bằng.

In [12]:
def evaluate_metrics_itemknn(
    eval_df,
    train_df,
    R,
    R_T_norm,
    user2idx,
    movie2idx,
    idx2movie,
    k_values=(5, 10, 20, 50),
    k_neighbours=K_NEIGHBOURS,
):
    """
    Tính Precision@K, Recall@K, NDCG@K, HR@K, MRR@K cho nhiều K.

    Parameters
    ----------
    eval_df     : DataFrame — tập đánh giá (valid / test / test_warm / test_cold)
    train_df    : DataFrame — dùng để lọc items đã xem
    R           : csr_matrix — binary interaction matrix (train)
    R_T_norm    : csr_matrix — L2-normalized item-user matrix
    user2idx    : dict — user_id → row index
    movie2idx   : dict — movie_id → col index
    idx2movie   : dict — col index → movie_id
    k_values    : tuple of int
    k_neighbours: int — KNN neighbourhood size

    Returns
    -------
    dict {K: {metric: float}}
    """
    if eval_df is None or len(eval_df) == 0:
        return {k: {"precision": 0., "recall": 0., "ndcg": 0.,
                    "hr": 0., "mrr": 0.} for k in k_values}

    max_k  = max(k_values)
    accum  = {k: {"precision": [], "recall": [], "ndcg": [],
                  "hr": [], "mrr": []} for k in k_values}

    # Pre-build: per user → set of train movie_ids
    train_seen = train_df.groupby("id")["movie_id"].apply(set).to_dict()

    # Per user → ground-truth item (từ eval_df)
    # LOO: mỗi user có đúng 1 item trong eval
    eval_by_user = eval_df.groupby("id")["movie_id"].apply(set).to_dict()

    n_movies = R_T_norm.shape[0]

    for user_id, true_movie_ids in eval_by_user.items():
        if user_id not in user2idx:
            # User không có trong train → bỏ qua
            continue

        u_idx = user2idx[user_id]

        # Tính scores cho tất cả items
        scores = get_user_scores_itemknn(u_idx, R, R_T_norm, k_neighbours)

        # Mask items đã thấy trong train (set score = -inf)
        seen_ids = train_seen.get(user_id, set())
        for mid in seen_ids:
            if mid in movie2idx:
                scores[movie2idx[mid]] = -np.inf

        # Lấy top max_k
        top_idx = np.argpartition(-scores, min(max_k, n_movies - 1))[:max_k]
        top_idx = top_idx[np.argsort(-scores[top_idx])]

        # Chuyển index về movie_id
        top_movie_ids = [idx2movie[i] for i in top_idx]

        # Relevance vector
        relevance  = np.array([1.0 if m in true_movie_ids else 0.0
                               for m in top_movie_ids])
        n_relevant = len(true_movie_ids)

        for k in k_values:
            rel_k  = relevance[:k]
            n_hits = float(rel_k.sum())

            precision = n_hits / k
            recall    = n_hits / n_relevant if n_relevant > 0 else 0.0
            hr        = 1.0 if n_hits > 0 else 0.0

            discounts = 1.0 / np.log2(np.arange(2, k + 2))
            dcg  = float((rel_k * discounts).sum())
            idcg = float(discounts[:min(n_relevant, k)].sum())
            ndcg = dcg / idcg if idcg > 0 else 0.0

            mrr = 0.0
            for j in range(k):
                if rel_k[j] > 0:
                    mrr = 1.0 / (j + 1)
                    break

            accum[k]["precision"].append(precision)
            accum[k]["recall"].append(recall)
            accum[k]["ndcg"].append(ndcg)
            accum[k]["hr"].append(hr)
            accum[k]["mrr"].append(mrr)

    summary = {}
    for k in k_values:
        summary[k] = {m: float(np.mean(v)) if v else 0.0
                      for m, v in accum[k].items()}
    return summary


def print_metrics(metrics, label):
    print(f"\n=== {label} ===")
    print(f"{'K':>4} | {'Prec':>8} | {'Recall':>8} | {'NDCG':>8} | {'HR':>8} | {'MRR':>8}")
    print("-" * 58)
    for k, m in sorted(metrics.items()):
        print(f"{k:>4} | {m['precision']:>8.4f} | {m['recall']:>8.4f} | "
              f"{m['ndcg']:>8.4f} | {m['hr']:>8.4f} | {m['mrr']:>8.4f}")


print("Metrics defined: evaluate_metrics_itemknn, print_metrics")

Metrics defined: evaluate_metrics_itemknn, print_metrics


## Mục 11 — Validation: lựa chọn `K_NEIGHBOURS`

Đánh giá trên tập validation để chọn `K_NEIGHBOURS` phù hợp.
Phạm vi thử nghiệm được giữ gọn nhằm phản ánh đúng tinh thần của một baseline.

In [13]:
# Đánh giá NDCG@10 trên valid set với các giá trị K khác nhau
K_CANDIDATES = [10, 20, 50, 100, 200]

print("Tuning K_NEIGHBOURS trên Valid set (NDCG@10)...")
print(f"{'K_neigh':>8} | {'NDCG@10':>10} | {'HR@10':>10}")
print("-" * 35)

best_k     = K_NEIGHBOURS
best_ndcg  = -1.0

for k_cand in K_CANDIDATES:
    vm = evaluate_metrics_itemknn(
        eval_df=valid_df,
        train_df=train_df,
        R=R,
        R_T_norm=R_T_norm,
        user2idx=user2idx,
        movie2idx=movie2idx,
        idx2movie=idx2movie,
        k_values=(10,),
        k_neighbours=k_cand,
    )
    ndcg10 = vm[10]["ndcg"]
    hr10   = vm[10]["hr"]
    print(f"{k_cand:>8} | {ndcg10:>10.4f} | {hr10:>10.4f}")

    if ndcg10 > best_ndcg:
        best_ndcg = ndcg10
        best_k    = k_cand

print(f"\n[ItemKNN] Best K_NEIGHBOURS = {best_k}  (Valid NDCG@10 = {best_ndcg:.4f})")
K_NEIGHBOURS = best_k  # Cập nhật

Tuning K_NEIGHBOURS trên Valid set (NDCG@10)...
 K_neigh |    NDCG@10 |      HR@10
-----------------------------------
      10 |     0.0061 |     0.0105
      20 |     0.0059 |     0.0105
      50 |     0.0058 |     0.0105
     100 |     0.0060 |     0.0127
     200 |     0.0062 |     0.0127

[ItemKNN] Best K_NEIGHBOURS = 200  (Valid NDCG@10 = 0.0062)


## Mục 12 — Đánh giá trên tập validation

In [14]:
print("[ItemKNN] Đánh giá trên VALID set...")
valid_metrics = evaluate_metrics_itemknn(
    eval_df=valid_df,
    train_df=train_df,
    R=R,
    R_T_norm=R_T_norm,
    user2idx=user2idx,
    movie2idx=movie2idx,
    idx2movie=idx2movie,
    k_values=K_VALUES,
    k_neighbours=K_NEIGHBOURS,
)
print_metrics(valid_metrics, "VALID")

[ItemKNN] Đánh giá trên VALID set...

=== VALID ===
   K |     Prec |   Recall |     NDCG |       HR |      MRR
----------------------------------------------------------
   5 |   0.0013 |   0.0063 |   0.0043 |   0.0063 |   0.0037
  10 |   0.0013 |   0.0127 |   0.0062 |   0.0127 |   0.0044
  20 |   0.0007 |   0.0148 |   0.0068 |   0.0148 |   0.0045
  50 |   0.0009 |   0.0443 |   0.0125 |   0.0443 |   0.0054


## Mục 13 — Đánh giá cuối trên tập kiểm thử

Đánh giá trên **Test**, **Test_Warm** và **Test_Cold** với giá trị `K_NEIGHBOURS` được chọn từ validation.

In [15]:
print("=" * 65)
print("FINAL EVALUATION — ItemKNN BASELINE")
print(f"Config: threshold={POSITIVE_THRESHOLD}, K_neighbours={K_NEIGHBOURS}")
print(f"Similarity: Cosine (binary implicit, no side information)")
print(f"Train items: {n_movies:,}  |  Train users: {n_users:,}")
print("=" * 65)

common_kwargs = dict(
    train_df=train_df,
    R=R,
    R_T_norm=R_T_norm,
    user2idx=user2idx,
    movie2idx=movie2idx,
    idx2movie=idx2movie,
    k_values=K_VALUES,
    k_neighbours=K_NEIGHBOURS,
)

for label, df in [
    ("VALID",     valid_df),
    ("TEST",      test_df),
    ("TEST_WARM", test_warm_df),
    ("TEST_COLD", test_cold_df),
]:
    print(f"\nĐánh giá {label}...")
    metrics = evaluate_metrics_itemknn(eval_df=df, **common_kwargs)
    print_metrics(metrics, label)

FINAL EVALUATION — ItemKNN BASELINE
Config: threshold=7, K_neighbours=200
Similarity: Cosine (binary implicit, no side information)
Train items: 75,115  |  Train users: 474

Đánh giá VALID...

=== VALID ===
   K |     Prec |   Recall |     NDCG |       HR |      MRR
----------------------------------------------------------
   5 |   0.0013 |   0.0063 |   0.0043 |   0.0063 |   0.0037
  10 |   0.0013 |   0.0127 |   0.0062 |   0.0127 |   0.0044
  20 |   0.0007 |   0.0148 |   0.0068 |   0.0148 |   0.0045
  50 |   0.0009 |   0.0443 |   0.0125 |   0.0443 |   0.0054

Đánh giá TEST...

=== TEST ===
   K |     Prec |   Recall |     NDCG |       HR |      MRR
----------------------------------------------------------
   5 |   0.0030 |   0.0148 |   0.0095 |   0.0148 |   0.0078
  10 |   0.0017 |   0.0169 |   0.0102 |   0.0169 |   0.0081
  20 |   0.0016 |   0.0316 |   0.0139 |   0.0316 |   0.0091
  50 |   0.0009 |   0.0464 |   0.0168 |   0.0464 |   0.0095

Đánh giá TEST_WARM...

=== TEST_WARM ===
 

## Mục 14 — Suy luận: gợi ý cho một người dùng

In [16]:
def recommend_for_user_itemknn(
    user_id, R, R_T_norm, movies_df,
    user2idx, movie2idx, idx2movie,
    train_df, k_neighbours=K_NEIGHBOURS, n_recs=10,
):
    """Gợi ý top-N movies cho 1 user bằng itemKNN, loại trừ phim đã xem trong train."""
    if user_id not in user2idx:
        print(f"User {user_id} không có trong train set.")
        return pd.DataFrame()

    u_idx  = user2idx[user_id]
    scores = get_user_scores_itemknn(u_idx, R, R_T_norm, k_neighbours)

    # Mask items đã xem
    seen_ids = set(train_df[train_df["id"] == user_id]["movie_id"].values)
    for mid in seen_ids:
        if mid in movie2idx:
            scores[movie2idx[mid]] = -np.inf

    top_indices   = np.argsort(-scores)[:n_recs]
    top_movie_ids = [idx2movie[i] for i in top_indices]

    result = movies_df[movies_df["id"].isin(top_movie_ids)][
        ["id", "original_title", "genres", "year_published", "rate"]
    ].copy()
    result["itemknn_score"] = [scores[movie2idx[mid]] for mid in result["id"]]
    return result.sort_values("itemknn_score", ascending=False)


# Demo
sample_user = train_df["id"].iloc[0]
print(f"[ItemKNN] Gợi ý top-10 cho user_id = {sample_user}\n")
recs = recommend_for_user_itemknn(
    user_id=sample_user,
    R=R, R_T_norm=R_T_norm,
    movies_df=movies_df,
    user2idx=user2idx, movie2idx=movie2idx, idx2movie=idx2movie,
    train_df=train_df,
    k_neighbours=K_NEIGHBOURS,
    n_recs=10,
)
print(recs.to_string(index=False))

[ItemKNN] Gợi ý top-10 cho user_id = 103007

    id        original_title                                          genres  year_published  rate  itemknn_score
173696        Shutter Island                                Thriller|Intriga          2010.0   7.6     139.981201
800220               Aladdin Animación|Fantástico|Musical|Infantil|Aventuras          1992.0   7.4     134.187805
531662                 Tesis                                Intriga|Thriller          1996.0   7.7     132.465607
381846 Das Leben der Anderen                                  Drama|Thriller          2006.0   8.0     131.760834
186830                   300              Acción|Bélico|Aventuras|Fantástico          2006.0   7.2     131.206085
779937          Nightcrawler                                        Thriller          2014.0   7.3     131.193619
519065          The Exorcist                                          Terror          1973.0   7.6     131.032288
721028       Rosemary's Baby               

---
## Tổng kết — So sánh ItemKNN, LightFM baseline và phương pháp đề xuất

| Thành phần | **ItemKNN (notebook này)** | **LightFM Baseline** | **Proposed Method** |
|-----------|--------------------------|----------------------|---------------------|
| **Split** | Leave-One-Out per user | Leave-One-Out per user | Leave-One-Out per user |
| **Interactions** | Binary implicit (rate ≥ 7) | Binary implicit + WARP | Binary implicit + WARP |
| **Similarity/Model** | Cosine item-item CF | LightFM (MF + features) | Proposed |
| **Đặc trưng nội dung item** | Không sử dụng | Thuộc tính có cấu trúc như genre, topic, country, year | Thuộc tính có cấu trúc kết hợp biểu diễn từ TinyBERT và KGE |
| **Đặc trưng người dùng** | Không sử dụng | Metadata hồ sơ người dùng | Metadata hồ sơ người dùng |
| **Xử lý cold-start item** | Không xử lý được | Xử lý ở mức hạn chế thông qua đặc trưng nội dung | Tốt hơn nhờ bổ sung biểu diễn ngữ nghĩa |
| **Xử lý cold-start user** | Không xử lý được | Xử lý ở mức hạn chế thông qua đặc trưng người dùng | Tốt hơn |
| **Tunable params** | K_NEIGHBOURS | components, lr, α, epochs | Nhiều hơn |
| **Metrics** | Precision, Recall, NDCG, HR, MRR @K | Giống | Giống |

> **Vai trò của notebook này trong bài báo:**
> ItemKNN là một **baseline collaborative filtering cổ điển**, không sử dụng bất kỳ side information nào.
> Nếu phương pháp đề xuất vượt **LightFM baseline** và **ItemKNN** trên **TEST_WARM**, có thể xem đây là bằng chứng cho thấy cả thành phần factorization lẫn nhóm đặc trưng bổ sung đều đóng góp vào hiệu quả cuối cùng.
> Trên **TEST_COLD**, kết quả của ItemKNN thường xấp xỉ 0; đây là điểm tham chiếu quan trọng để làm nổi bật lợi thế của các phương pháp có sử dụng đặc trưng nội dung hoặc biểu diễn ngữ nghĩa.